# One Expert Gets All the Traffic

**Topic:** Mixture-of-experts routing &nbsp;·&nbsp; **Difficulty:** Hard

A learned router is a positive feedback loop. The expert that receives slightly
more tokens gets slightly more gradient, becomes slightly better, and therefore
receives more tokens next step. Left alone this runs away: routing collapses onto
one or two experts and the other fourteen are dead weight you are still paying to
keep in memory.

Simulate the loop directly, watch it collapse, then add the standard auxiliary
load-balancing loss and watch it not.

## The data (created in the next cell)

- `base_logits` `(4096, 16)`, router logits for 4,096 tokens over 16 experts,
  from a fixed random router.
- Top-2 routing, 200 rounds of feedback, `eta = 2.0`.
- The load-balancing weight `lam` is `0.0` for the plain run and `2.0` for the
  balanced one.

## Your task

With **NumPy only**, maintain a per-expert bias vector, starting at zeros, and
repeat 200 times:

1. Compute `P = softmax(base_logits + bias)` and route each token to its
   top 2 experts.
2. `f`, the fraction of routed slots going to each expert (sums to 1).
3. `Pmean`, each expert's mean router probability over the batch.
4. Update `bias += eta * f - lam * n_experts * f * Pmean`. The first term is
   the feedback loop (busy experts get more attractive). The second is the
   standard auxiliary loss gradient, which is `n_experts * sum_e f_e * P_e`.

Run it with `lam = 0.0` and with `lam = 2.0`, then for each report `frac_max`
(largest expert's share of routed slots), `n_dead` (experts receiving nothing),
and `entropy` (bits, of the load distribution).

Set `answers` to a dict with `"frac_max_plain"`, `"n_dead_plain"`,
`"entropy_plain"`, `"frac_max_bal"`, `"n_dead_bal"`, `"entropy_bal"`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the plain run collapses onto a couple of experts and the balanced run
keeps every expert alive near the uniform 4-bit maximum.

In [ ]:
import numpy as np

rng = np.random.default_rng(4007)
n_tokens, d, n_exp = 4096, 64, 16
X = rng.normal(0, 1, (n_tokens, d))
W_r = np.random.default_rng(66).normal(0, 0.1, (d, n_exp))
base_logits = X @ W_r
top_k, rounds, eta = 2, 200, 2.0
print(f"{n_tokens} tokens, {n_exp} experts, top-{top_k} routing, {rounds} rounds")
print(f"uniform load would be {1 / n_exp:.4f} per expert "
      f"({np.log2(n_exp):.1f} bits of entropy)")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("frac_max_plain", [], 0.5, -0.08371900740724958, 1e-06),
    ("n_dead_plain", [], 14.0, -2.344132207402988, 1e-06),
    ("entropy_plain", [], 1.0, -0.16743801481449916, 1e-06),
    ("frac_max_bal", [], 0.0732421875, -0.012263526475671326, 1e-06),
    ("n_dead_bal", [], 0.0, 0.0, 1e-06),
    ("entropy_bal", [], 3.99428737780451, -0.6687955491381985, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "frac_max_plain" in answers:
    _collapse = answers["frac_max_plain"] > 0.3 and answers["n_dead_plain"] >= 10
    _fixed = answers["n_dead_bal"] == 0 and answers["entropy_bal"] > 3.9
    print(f"  {'plain router collapses':<24} {'ok' if _collapse else 'wrong'}  "
          f"(top expert {answers['frac_max_plain']:.1%}, "
          f"{int(answers['n_dead_plain'])} of 16 dead, {answers['entropy_plain']:.2f} bits)")
    print(f"  {'balancing keeps all alive':<24} {'ok' if _fixed else 'wrong'}  "
          f"(top expert {answers['frac_max_bal']:.1%}, "
          f"{int(answers['n_dead_bal'])} dead, {answers['entropy_bal']:.2f} bits)")
    _ok = _ok and _collapse and _fixed

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

The auxiliary term is the product of two quantities that both rise when an expert
is overused (its share of the load and its mean router probability), so it is
minimized at uniform load and its gradient pushes probability away from busy
experts. The factor `n_experts` normalizes it so the balanced value is 1 whatever
the expert count. Note that it is a *soft* constraint: it shapes the router but
cannot guarantee anything, which is why production MoE layers pair it with a hard
capacity factor and drop the overflow.

```python
import numpy as np

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def simulate(lam):
    bias = np.zeros(n_exp)
    for _ in range(rounds):
        P = softmax(base_logits + bias)
        idx = np.argsort(-P, axis=1)[:, :top_k]
        M = np.zeros_like(P)
        np.put_along_axis(M, idx, 1.0, axis=1)
        f = M.sum(axis=0) / (n_tokens * top_k)
        Pmean = P.mean(axis=0)
        bias = bias + eta * f - lam * n_exp * f * Pmean

    P = softmax(base_logits + bias)
    idx = np.argsort(-P, axis=1)[:, :top_k]
    counts = np.bincount(idx.ravel().astype(np.intp), minlength=n_exp).astype(float)
    share = counts / counts.sum()
    nz = share[share > 0]
    return (float(share.max()), float((counts == 0).sum()),
            float(-(nz * np.log2(nz)).sum()))

fm_p, nd_p, en_p = simulate(0.0)
fm_b, nd_b, en_b = simulate(2.0)

answers = {
    "frac_max_plain": fm_p, "n_dead_plain": nd_p, "entropy_plain": en_p,
    "frac_max_bal": fm_b, "n_dead_bal": nd_b, "entropy_bal": en_b,
}
```